# Customer Shopping Behaviour Analysis
**Python → Databricks SQL → Power BI** | 3,900 customer purchase records

Portfolio version of the original Databricks notebook, with verbose execution output removed. Raw CSV is not included: place `customer_shopping_behavior.csv` in a location accessible to your Databricks notebook and adjust the path. The business report explains assumptions and limitations.

## 1. Load and inspect the dataset

In [ ]:
import pandas as pd

# Replace this with the accessible file path in Databricks.
csv_path = "customer_shopping_behavior.csv"
df = pd.read_csv(csv_path)
print("Rows, columns:", df.shape)
print(df.info())
display(df.head())

In [ ]:
# Missing values and basic validity checks
print(df.isna().sum().sort_values(ascending=False))
print(df.describe(include="all"))
assert df["Customer ID"].is_unique, "Each expected Customer ID should occur once"

## 2. Clean and engineer features
The 37 missing review ratings are filled with each category’s median. The quartile-based age groups are **descriptive bins** rather than an official demographic classification.

In [ ]:
# Impute missing ratings using the median within each product category
df["Review Rating"] = (
    df.groupby("Category")["Review Rating"]
      .transform(lambda s: s.fillna(s.median()))
)

# Standardise source column names
df.columns = (df.columns.str.lower()
                         .str.strip()
                         .str.replace(r"[^a-z0-9]+", "_", regex=True)
                         .str.strip("_"))
df = df.rename(columns={"purchase_amount_usd": "purchase_amount"})

# Age quartiles (with equal ages allowed at shared bin boundaries)
df["age_group"] = pd.qcut(
    df["age"], q=4, labels=["Young", "Adult", "Middle Aged", "Senior"]
)

# Flag redundancy without silently removing a source field
print("Discount and promo indicators match:",
      df["discount_applied"].equals(df["promo_code_used"]))
print("Missing ratings after treatment:", df["review_rating"].isna().sum())
display(df.head())

## 3. Validate key metrics

In [ ]:
kpis = {
    "purchase_records": len(df),
    "unique_customer_ids": df["customer_id"].nunique(),
    "purchase_value_usd": float(df["purchase_amount"].sum()),
    "avg_purchase_usd": round(float(df["purchase_amount"].mean()), 2),
    "avg_review_rating": round(float(df["review_rating"].mean()), 2),
}
print(kpis)
assert kpis["purchase_records"] == 3900
assert kpis["purchase_value_usd"] == 233081.0

## 4. Register a temporary view for SQL

In [ ]:
# Databricks runtime: converts the Pandas DataFrame to Spark
spark_df = spark.createDataFrame(df)
spark_df.createOrReplaceTempView("customer")
print("Temporary SQL view created: customer")

In [ ]:
%sql
SELECT category, COUNT(*) AS purchase_records,
       SUM(purchase_amount) AS purchase_value_usd
FROM customer
GROUP BY category
ORDER BY purchase_value_usd DESC

In [ ]:
%sql
WITH item_counts AS (
    SELECT category, item_purchased, COUNT(*) AS purchase_records
    FROM customer
    GROUP BY category, item_purchased
), ranked_items AS (
    SELECT *, ROW_NUMBER() OVER (
        PARTITION BY category
        ORDER BY purchase_records DESC, item_purchased ASC
    ) AS item_rank
    FROM item_counts
)
SELECT * FROM ranked_items
WHERE item_rank <= 3
ORDER BY category, item_rank

## 5. Persist a Delta table for Power BI
The first command is safe to rerun without overwriting an existing table. To **refresh** the persistent table after intentional changes, use `CREATE OR REPLACE TABLE` only after confirming the target table and its permissions.

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS workspace.default.customer_powerbi
USING DELTA
AS SELECT * FROM customer

In [ ]:
%sql
SELECT COUNT(*) AS purchase_records
FROM workspace.default.customer_powerbi

## 6. Power BI and SQL assets
- `sql/customer_shopping_analysis.sql` contains the full set of readable analytics queries.
- `dashboard/Customer_Behavior_Analysis_Dashboard.pbix` contains the Power BI dashboard.
- `dashboard/dashboard_preview.png` is the dashboard screenshot extracted from the supplied reference report.

**Interpretation cautions:** This sample is cross-sectional, ratings include category-median imputation, and observed subscription/discount differences do not identify causal effects. No margin or time-period data was provided.